# Augusto Preto

# Atividade Prática 1 - Compiladores
## Analisadores léxicos com Lark e desafio RastreioLang

Execute as células em ordem. Este notebook contém todas as implementações e
testes; não exige outros arquivos. As interfaces interativas funcionam no
Colab ou em um kernel Jupyter com ipywidgets.

**Organização:** preparação, código, respostas teóricas, experimentos,
interfaces A1/A2/B1/B2, desafio, testes e roteiro de apresentação.



In [1]:
import importlib.metadata
import subprocess
import sys

necessarias = {"lark": "1.3.1", "ipywidgets": "8.1.8"}
instalar = []
for pacote, versao in necessarias.items():
    try:
        atual = importlib.metadata.version(pacote)
    except importlib.metadata.PackageNotFoundError:
        atual = None
    if atual != versao:
        instalar.append(f"{pacote}=={versao}")
if instalar:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *instalar])

try:
    from google.colab import output
except ImportError:
    pass
else:
    output.enable_custom_widget_manager()

print("Ambiente pronto: Lark e ipywidgets.")


Ambiente pronto: Lark e ipywidgets.


## Núcleo e gramáticas dos exercícios

In [2]:
from collections import Counter, defaultdict
from datetime import datetime
from decimal import Decimal, ROUND_HALF_UP
from functools import lru_cache
import html
import re

from lark import Lark
from lark.exceptions import UnexpectedCharacters


def compilar(definicoes, comentarios=True):
    nomes = [linha.split(':', 1)[0].split('.')[0] for linha in definicoes.splitlines()
             if linha.strip() and not linha.lstrip().startswith('//')]
    ruido = '\n%ignore /[ \\t\\r\\n]+/\n'
    if comentarios:
        ruido += 'COMENTARIO: /#[^\\n]*/\n%ignore COMENTARIO\n'
    return 'start: _token*\n_token: ' + ' | '.join(nomes) + '\n' + definicoes + ruido


@lru_cache(maxsize=16)
def criar_lexer(gramatica):
    return Lark(gramatica, parser='lalr', lexer='basic', propagate_positions=True)


def analisar(texto, gramatica, dominio='geral'):
    try:
        return list(criar_lexer(gramatica).lex(texto))
    except UnexpectedCharacters as erro:
        resto = texto[erro.pos_in_stream:].split('\n', 1)[0]
        if erro.char == '"':
            dica = 'Feche as aspas do nome, status ou descrição antes de mudar de linha.'
        elif dominio == 'rastreio':
            if re.match(r'\d', resto):
                dica = ('CEP: 01310-100; data: 10/09/2026; hora: 08:15; '
                        'peso: 1,250kg. Confira os separadores e o tamanho do campo.')
            elif '@' in resto.split(' ', 1)[0] or erro.char == '@':
                dica = 'Contato de entrega deve ser um e-mail como em@exemplo.com.'
            else:
                dica = ('Código de rastreio: duas letras, nove dígitos e duas letras '
                        '(BR123456789BR). Status e endereços precisam de aspas.')
        elif dominio == 'banco':
            dica = ('Confira a chave: nome@dominio.com, CPF 123.456.789-09, '
                    'CNPJ 12.345.678/0001-90 ou telefone no formato desta versão. '
                    'Valores: R$ 10,50; datas: 01/09/2026.')
        else:
            dica = ('Preço usa vírgula e dois centavos (R$ 25,90); quantidade usa x '
                    '(2x). A1 exige maiúsculas nos comandos e R$ no preço.')
        erro.dica = dica
        raise


DEFS_A1 = r'''
PEDIDO: "PEDIDO"
ENTREGA: "ENTREGA"
CUPOM: "CUPOM"
QTD.2: /\d+x/
ITEM: /"[^"\n]*"/
PRECO.2: /R\$ ?\d+,\d{2}/
CODIGO: /[A-Z][A-Z0-9]*/
'''
GRAMATICA_A1_BASE = compilar(DEFS_A1, comentarios=False)
GRAMATICA_A1 = compilar(DEFS_A1 + '\nOBS: "OBS"\n', comentarios=False)

DEFS_A2 = r'''
PEDIDO.3: /pedido\b/i
ENTREGA.3: /entrega\b/i
CUPOM.3: /cupom\b/i
OBS.3: /obs\b/i
TAXA.3: /taxa\b/i
PAGAMENTO.3: /pagamento\b/i
FORMA_PGTO.3: /(pix|cart[aã]o|dinheiro|vale_refeicao)\b/i
QTD.2: /\d+x\b/i
PRECO.2: /R\$ ?\d{1,3}(\.\d{3})*,\d{2}/
TEXTO: /"[^"\n]*"/
NUMERO: /\d+/
CODIGO: /[A-Za-z_][A-Za-z0-9_]*/
'''
GRAMATICA_A2_BASE = compilar(DEFS_A2)
# PRECO.2 vence NUMERO.0 no prefixo 25 de 25,90.
# QTD.2 e PRECO.2 não casam a mesma entrada completa: x versus vírgula.
DEFS_A2_FINAL = DEFS_A2.replace('vale_refeicao)', 'vale_refeicao|vr)').replace(
    r'/R\$ ?\d{1,3}(\.\d{3})*,\d{2}/',
    r'/(R\$ ?)?\d{1,3}(\.\d{3})*,\d{2}(?![\w,.])/')
GRAMATICA_A2 = compilar(DEFS_A2_FINAL)
CUPONS = {'DESC10': Decimal('0.10'), 'DESC15': Decimal('0.15'),
          'DESC20': Decimal('0.20'), 'PRIMEIRACOMPRA': Decimal('0.20')}
CUPONS_FRETE = {'FRETEGRATIS', 'FRETE10'}

DEFS_B1 = r'''
PIX.3: /pix\b/i
DIRECAO.3: /(enviado|recebido)\b/i
PARA.3: /para\b/i
DE.3: /de\b/i
EM.3: /em\b/i
VALOR.2: /R\$ ?\d{1,3}(\.\d{3})*,\d{2}/
DATA.2: /\d{2}\/\d{2}\/\d{4}/
HORA.2: /\d{2}:\d{2}/
CHAVE_EMAIL.4: /[a-z0-9._+-]+@[a-z0-9-]+(\.[a-z0-9-]+)+/i
CHAVE_CPF.2: /\d{3}\.\d{3}\.\d{3}-\d{2}/
CHAVE_TELEFONE.2: /\+55\d{10,11}/
CHAVE_ALEATORIA.2: /[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}/i
'''
DEFINICAO_CNPJ = r'CHAVE_CNPJ.2: /\d{2}\.\d{3}\.\d{3}\/\d{4}-\d{2}/'
GRAMATICA_B1_BASE = compilar(DEFS_B1, comentarios=False)
GRAMATICA_B1 = compilar(DEFS_B1 + '\n' + DEFINICAO_CNPJ, comentarios=False)
DEFS_B2 = DEFS_B1.replace('(enviado|recebido)', '(enviado|recebido|pago)') + '\n' + DEFINICAO_CNPJ + r'''
SALDO_INICIAL.3: /saldo\s+inicial\b/i
TED.3: /ted\b/i
BOLETO.3: /boleto\b/i
TARIFA.3: /tarifa\b/i
DESCRICAO: /"[^"\n]*"/
'''
GRAMATICA_B2_BASE = compilar(DEFS_B2)
# 11 dígitos locais são telefone por convenção DESTA linguagem.
# Não existe CPF sem pontuação nesta versão: a prioridade não descobriria a intenção.
GRAMATICA_B2 = compilar(DEFS_B2.replace(
    r'/\+55\d{10,11}/', r'/(\+55\d{10,11}|\d{11})(?![\w+])/')
    + r'\nSAQUE.3: /saque\b/i'.replace(r'\n', '\n'))


def tokenizar_a1(texto):
    return analisar(texto, GRAMATICA_A1, 'pedido')


def tokenizar_a2(texto):
    return analisar(texto, GRAMATICA_A2, 'pedido')


def tokenizar_b1(texto, prioridade_email=4):
    gramatica = GRAMATICA_B1.replace('CHAVE_EMAIL.4', f'CHAVE_EMAIL.{prioridade_email}')
    return analisar(texto, gramatica, 'banco')


def tokenizar_b2(texto):
    return analisar(texto, GRAMATICA_B2, 'banco')


def valor_em_reais(lexema):
    return Decimal(str(lexema).replace('R$', '').strip().replace('.', '').replace(',', '.'))


def brl(valor):
    numero = f'{abs(valor):,.2f}'.replace(',', '_').replace('.', ',').replace('_', '.')
    return ('- ' if valor < 0 else '') + 'R$ ' + numero


def linhas_de_tokens(tokens):
    linhas = defaultdict(list)
    for token in tokens:
        linhas[token.line].append(token)
    return linhas


def comanda(tokens):
    itens, observacoes, avisos = [], [], []
    taxa, cupom, pagamento, endereco = Decimal(0), None, 'não informado', 'retirada'
    for linha, ts in linhas_de_tokens(tokens).items():
        tipos = [t.type for t in ts]
        if tipos[:4] == ['PEDIDO', 'QTD', 'TEXTO', 'PRECO']:
            if tipos[4:] not in ([], ['OBS', 'TEXTO']):
                raise ValueError(f'Linha {linha}: após o item, use OBS "texto" ou termine a linha.')
            itens.append((int(ts[1][:-1]), str(ts[2])[1:-1], valor_em_reais(ts[3])))
            if len(ts) > 4:
                observacoes.append(str(ts[5])[1:-1])
        elif tipos == ['TAXA', 'PRECO']:
            taxa = valor_em_reais(ts[1])
        elif tipos == ['CUPOM', 'CODIGO']:
            cupom = str(ts[1]).upper()
        elif tipos == ['PAGAMENTO', 'FORMA_PGTO']:
            pagamento = str(ts[1]).upper()
        elif tipos == ['ENTREGA', 'TEXTO']:
            endereco = str(ts[1])[1:-1]
        elif tipos == ['OBS', 'TEXTO']:
            observacoes.append(str(ts[1])[1:-1])
        else:
            raise ValueError(f'Linha {linha}: sequência não reconhecida pela comanda.')
    subtotal = sum((q*p for q, _, p in itens), Decimal(0))
    desconto = (subtotal * CUPONS.get(cupom, Decimal(0))).quantize(Decimal('.01'), rounding=ROUND_HALF_UP)
    if cupom in CUPONS_FRETE:
        taxa = Decimal(0)
    if cupom and cupom not in CUPONS and cupom not in CUPONS_FRETE:
        avisos.append(f'Cupom {cupom} inexistente: desconto recusado pela regra de negócio.')
    return dict(itens=itens, subtotal=subtotal, desconto=desconto, taxa=taxa,
                total=subtotal-desconto+taxa, pagamento=pagamento, endereco=endereco,
                observacoes=observacoes, avisos=avisos)


def conciliar(tokens):
    saldo_inicial, transacoes = Decimal(0), []
    viu_saldo = False
    for linha, ts in linhas_de_tokens(tokens).items():
        tipos = [t.type for t in ts]
        if tipos == ['SALDO_INICIAL', 'VALOR']:
            if viu_saldo or transacoes:
                raise ValueError('SALDO INICIAL deve aparecer uma única vez, antes das transações.')
            saldo_inicial, viu_saldo = valor_em_reais(ts[1]), True
            continue
        op = tipos[0]
        chave = None
        if op in ('PIX', 'TED'):
            if len(ts) < 8 or tipos[:3] != [op, 'DIRECAO', 'VALOR'] or not tipos[4].startswith('CHAVE_'):
                raise ValueError(f'Linha {linha}: use {op} ENVIADO/RECEBIDO valor PARA/DE chave EM data hora.')
            direcao = str(ts[1]).upper()
            if direcao not in ('ENVIADO', 'RECEBIDO') or tipos[3] != ('DE' if direcao == 'RECEBIDO' else 'PARA'):
                raise ValueError(f'Linha {linha}: direção incompatível com PARA/DE.')
            valor, chave, resto = valor_em_reais(ts[2]), ts[4], ts[5:]
            sinal = 1 if direcao == 'RECEBIDO' else -1
        elif op == 'BOLETO' and tipos[:3] == ['BOLETO', 'DIRECAO', 'VALOR'] and str(ts[1]).upper() == 'PAGO':
            valor, resto, sinal, direcao = valor_em_reais(ts[2]), ts[3:], -1, 'PAGO'
        elif op in ('SAQUE', 'TARIFA') and tipos[:2] == [op, 'VALOR']:
            valor, resto, sinal, direcao = valor_em_reais(ts[1]), ts[2:], -1, 'DÉBITO'
        else:
            raise ValueError(f'Linha {linha}: operação ou sequência de campos inválida.')
        if [t.type for t in resto] not in (['EM', 'DATA', 'HORA'], ['EM', 'DATA', 'HORA', 'DESCRICAO']):
            raise ValueError(f'Linha {linha}: termine com EM data hora e descrição opcional entre aspas.')
        instante = datetime.strptime(f'{resto[1]} {resto[2]}', '%d/%m/%Y %H:%M')
        transacoes.append(dict(linha=linha, tipo=op, direcao=direcao, valor=valor*sinal,
                               chave=chave, data=str(resto[1]), hora=str(resto[2]), instante=instante,
                               descricao=str(resto[3])[1:-1] if len(resto) == 4 else ''))
    return saldo_inicial, transacoes


def alertas(transacoes):
    avisos = []
    for tr in transacoes:
        if tr['tipo'] != 'PIX' or tr['valor'] >= 0:
            continue
        hora = tr['instante'].hour
        if (hora >= 20 or hora < 6) and abs(tr['valor']) > 1000:
            avisos.append(f"Linha {tr['linha']}: PIX noturno acima do limiar didático de R$ 1.000,00.")
        if tr['chave'].type == 'CHAVE_ALEATORIA' and abs(tr['valor']) > 500:
            avisos.append(f"Linha {tr['linha']}: PIX para chave aleatória acima de R$ 500,00.")
    return avisos

## Desafio para entrega: RastreioLang

In [3]:
# Prioridades: EMAIL.5 > reservadas.4 > código.3 > literais.2 > identificador.0.
# EMAIL precisa vencer EM em em@exemplo.com: @ cria uma fronteira \b.
# COD_RASTREIO precisa vencer IDENTIFICADOR, que também aceita BR123456789BR.
DEFS_RASTREIO = r'''
RASTREIO.4: /rastreio\b/i
STATUS.4: /status\b/i
CEP.4: /cep\b/i
EM.4: /em\b/i
ORIGEM.4: /origem\b/i
DESTINO.4: /destino\b/i
PESO.4: /peso\b/i
FRETE.4: /frete\b/i
CONTATO.4: /contato\b/i
PRAZO.4: /prazo\b/i
DESTINATARIO.4: /destinatario\b/i
EMAIL.5: /[a-z0-9._+-]+@[a-z0-9-]+(\.[a-z0-9-]+)+(?=$|[\s#])/i
COD_RASTREIO.3: /[A-Z]{2}\d{9}[A-Z]{2}(?=$|[\s#])/i
CEP_VALOR.2: /\d{5}-\d{3}(?=$|[\s#])/
DATA.2: /\d{2}\/\d{2}\/\d{4}(?=$|[\s#])/
HORA.2: /\d{2}:\d{2}(?=$|[\s#])/
PESO_VALOR.2: /\d+(,\d{1,3})?kg(?=$|[\s#])/i
VALOR.2: /R\$ ?\d{1,3}(\.\d{3})*,\d{2}(?=$|[\s#])/
CPF_VALOR.2: /\d{3}\.\d{3}\.\d{3}-\d{2}(?=$|[\s#])/
TEXTO: /"[^"\n]*"/
NUMERO: /\d+(?=$|[\s#])/
IDENTIFICADOR: /[A-Za-z_][A-Za-z0-9_]*(?=$|[\s#])/
'''
GRAMATICA_RASTREIO = compilar(DEFS_RASTREIO)


def tokenizar_rastreio(texto):
    return analisar(texto, GRAMATICA_RASTREIO, 'rastreio')


def resumo_rastreio(tokens):
    eventos = []
    cabecalho = ['RASTREIO', 'COD_RASTREIO', 'STATUS', 'TEXTO', 'CEP', 'CEP_VALOR', 'EM', 'DATA', 'HORA']
    opcionais = dict(ORIGEM='TEXTO', DESTINO='TEXTO', PESO='PESO_VALOR', FRETE='VALOR',
                    CONTATO='EMAIL', PRAZO='NUMERO', DESTINATARIO='CPF_VALOR')
    for linha, ts in linhas_de_tokens(tokens).items():
        if [t.type for t in ts[:9]] != cabecalho:
            raise ValueError(f'Linha {linha}: use RASTREIO código STATUS "texto" CEP número EM data hora.')
        try:
            instante = datetime.strptime(f'{ts[7]} {ts[8]}', '%d/%m/%Y %H:%M')
        except ValueError as erro:
            raise ValueError(f'Linha {linha}: data ou hora inexistente no calendário.') from erro
        evento = dict(linha=linha, codigo=str(ts[1]).upper(), status=str(ts[3])[1:-1],
                      cep=str(ts[5]), instante=instante)
        resto = ts[9:]
        if len(resto) % 2:
            raise ValueError(f'Linha {linha}: campo opcional sem valor.')
        for campo, valor in zip(resto[::2], resto[1::2]):
            if opcionais.get(campo.type) != valor.type or campo.type.lower() in evento:
                raise ValueError(f'Linha {linha}: campo {campo} repetido ou com tipo incorreto.')
            convertido = str(valor)
            if campo.type == 'FRETE':
                convertido = valor_em_reais(valor)
            elif campo.type == 'PESO':
                convertido = Decimal(str(valor)[:-2].replace(',', '.'))
            elif campo.type == 'PRAZO':
                convertido = int(str(valor))
            elif valor.type == 'TEXTO':
                convertido = str(valor)[1:-1]
            evento[campo.type.lower()] = convertido
        eventos.append(evento)
    fretes = {}
    for evento in sorted(eventos, key=lambda e: e['instante']):
        if 'frete' in evento:
            fretes[evento['codigo']] = evento['frete']
    return dict(eventos=eventos, encomendas=len({e['codigo'] for e in eventos}),
                frete_total=sum(fretes.values(), Decimal(0)))

## Casos de uso (todos os dados são exemplos didáticos)

In [4]:
UUID_EXEMPLO = '7d9f3c2a-1b4e-4c8a-9f21-0a6b5e3d7c10'
EXTRATO_EXEMPLO = '''# Extrato de demonstração
SALDO INICIAL R$ 2.500,00
PIX RECEBIDO R$ 3.200,00 DE 12.345.678/0001-90 EM 05/09/2026 09:15 "Salário"
PIX ENVIADO R$ 1.200,00 PARA maria.souza@example.com EM 06/09/2026 10:02 "Aluguel"
PIX ENVIADO R$ 89,90 PARA +5511987654321 EM 07/09/2026 19:45 "Pizza"
BOLETO PAGO R$ 149,90 EM 08/09/2026 08:30 "Internet"
PIX RECEBIDO R$ 50,00 DE 123.456.789-09 EM 09/09/2026 12:10 "Almoço"
PIX ENVIADO R$ 35,00 PARA 7d9f3c2a-1b4e-4c8a-9f21-0a6b5e3d7c10 EM 10/09/2026 14:32 "Doação"
TED ENVIADO R$ 500,00 PARA 98.765.432/0001-10 EM 10/09/2026 16:00 "Curso"
TARIFA R$ 12,50 EM 10/09/2026 23:59 "Serviços"'''
CASOS_A1 = {
    'Guiado: OBS': 'PEDIDO 1x "Pizza" R$ 50,00 OBS "sem cebola"',
    'Pedido com entrega': 'PEDIDO 3x "Coxinha" R$ 7,50 ENTREGA "Rua A, 100"',
    'Palavra maior': 'PEDIDOS 2x',
    'Inválido: minúsculas': 'pedido 2x',
    'Inválido: símbolo': 'PEDIDO 2x "Açaí 500ml" R$ 19,90 @CUPOM',
}
CASOS_A2 = {
    'Guiado: VR e DESC20': 'PEDIDO 2x "Pizza" R$ 50,00\nTAXA R$ 10,00\nCUPOM DESC20\nPAGAMENTO vr',
    'Desafio: sem R$': 'pedido 3X "Pastel" 8,50\npagamento cartão',
    'Pedido completo': 'PEDIDO 2x "X-Burger" R$ 25,90\nPEDIDO 1x "Batata" R$ 18,50\nPEDIDO 3x "Refri" R$ 6,00\nTAXA R$ 7,99\nCUPOM DESC10\nPAGAMENTO pix',
    'Frete grátis': 'pedido 1x "Combo" R$ 1.150,00\ntaxa R$ 15,00\ncupom FRETEGRATIS',
    'Cupom inexistente (léxico válido)': 'PEDIDO 1x "Temaki" R$ 32,00\nCUPOM GANHEI100',
    'Inválido: centavos': 'PEDIDO 2x "Pastel" 8.50',
    'Inválido: aspas': 'PEDIDO 1x "Pizza',
}
CASOS_B1 = {
    'Guiado: CNPJ': 'PIX RECEBIDO R$ 100,00 DE 12.345.678/0001-90 EM 10/09/2026 08:15',
    'Conflito pix@': 'PIX ENVIADO R$ 10,00 PARA pix@loja.com.br EM 01/09/2026 08:00',
    'CPF': 'PIX RECEBIDO R$ 50,00 DE 123.456.789-09 EM 09/09/2026 12:10',
    'Telefone': 'PIX ENVIADO R$ 89,90 PARA +5511987654321 EM 07/09/2026 19:45',
    'Aleatória': f'PIX ENVIADO R$ 35,00 PARA {UUID_EXEMPLO} EM 10/09/2026 14:32',
    'Inválido: telefone local no B1': 'PIX ENVIADO R$ 20,00 PARA 11987654321 EM 02/09/2026 10:10',
}
CASOS_B2 = {
    'Extrato completo': EXTRATO_EXEMPLO,
    'Guiado: SAQUE': 'SALDO INICIAL R$ 300,00\nSAQUE R$ 200,00 EM 05/09/2026 18:00',
    'Desafio: telefone local': 'PIX ENVIADO R$ 20,00 PARA 11987654321 EM 02/09/2026 10:10',
    'Desafio: alerta de chave aleatória': f'PIX ENVIADO R$ 600,00 PARA {UUID_EXEMPLO} EM 10/09/2026 14:32',
    'Alerta noturno': 'PIX ENVIADO R$ 1.500,00 PARA a@example.com EM 11/09/2026 22:40',
    'Inválido: e-mail': 'SALDO INICIAL R$ 300,00\nPIX ENVIADO R$ 45,00 PARA joao@@mail.com EM 01/09/2026 10:00',
    'Inválido: aspas': 'PIX ENVIADO R$ 15,00 PARA a@b.com EM 01/09/2026 10:00 "Café',
}
CASOS_RASTREIO = {
    'Válido 1: entrega': 'RASTREIO BR123456789BR STATUS "saiu para entrega" CEP 01310-100 EM 10/09/2026 08:15',
    'Válido 2: completo e minúsculas': '# Encomenda de exemplo\nrastreio AB987654321BR status "em trânsito" cep 20040-020 em 11/09/2026 09:30 origem "São Paulo" destino "Rio de Janeiro" peso 1,250kg frete R$ 25,90 contato em@exemplo.com prazo 3 destinatario 123.456.789-09',
    'Válido 3: vários eventos': 'RASTREIO BR123456789BR STATUS "postado" CEP 01310-100 EM 10/09/2026 08:15 FRETE R$ 25,90\nRASTREIO BR123456789BR STATUS "entregue" CEP 01310-100 EM 12/09/2026 14:30 FRETE R$ 25,90\nRASTREIO CD111222333BR STATUS "postado" CEP 30130-010 EM 12/09/2026 15:00 FRETE R$ 10,00',
    'Inválido 1: aspas abertas': 'RASTREIO BR123456789BR STATUS "saiu para entrega',
    'Inválido 2: CEP com ponto': 'RASTREIO BR123456789BR STATUS "postado" CEP 01310.100 EM 10/09/2026 08:15',
    'Inválido 3: contato': 'RASTREIO BR123456789BR STATUS "postado" CEP 01310-100 EM 10/09/2026 08:15 CONTATO em@@exemplo.com',
}

## Interface compartilhada

In [5]:
def mascarar(token):
    valor = str(token)
    if token.type in ('CHAVE_EMAIL', 'EMAIL'):
        return valor[0] + '***@' + valor.split('@', 1)[1]
    if token.type in ('CHAVE_CPF', 'CPF_VALOR'):
        return '***.***.***-' + valor[-2:]
    if token.type == 'CHAVE_CNPJ':
        return '**.***.***/****-' + valor[-2:]
    if token.type == 'CHAVE_TELEFONE':
        return '*' * (len(valor)-4) + valor[-4:]
    if token.type == 'CHAVE_ALEATORIA':
        return valor[:8] + '-****'
    return valor


def cor_token(tipo):
    if tipo.startswith('CHAVE') or tipo in ('EMAIL', 'CPF_VALOR'):
        return '#9d174d'
    if tipo in ('PRECO', 'VALOR', 'PESO_VALOR'):
        return '#166534'
    if tipo in ('ITEM', 'TEXTO', 'DESCRICAO'):
        return '#9a3412'
    if tipo in ('NUMERO', 'QTD', 'DATA', 'HORA', 'CEP_VALOR', 'COD_RASTREIO'):
        return '#6b21a8'
    return '#1e40af'


def tabela_html(cabecalho, linhas):
    h = '<tr>' + ''.join(f'<th>{html.escape(str(c))}</th>' for c in cabecalho) + '</tr>'
    corpo = ''.join('<tr>' + ''.join(f'<td>{html.escape(str(v))}</td>' for v in linha) + '</tr>' for linha in linhas)
    return '<div style="overflow:auto"><table class="p1-table">' + h + corpo + '</table></div>'


def tabela_tokens_html(tokens, ocultar=False):
    return tabela_html(['#', 'TOKEN', 'LEXEMA', 'LINHA', 'COLUNA'],
                      [(i, t.type, mascarar(t) if ocultar else str(t), t.line, t.column)
                       for i, t in enumerate(tokens, 1)])


def texto_colorido_html(texto, tokens, ocultar=False):
    partes, cursor = [], 0
    for token in tokens:
        intervalo = texto[cursor:token.start_pos]
        if ocultar:
            intervalo = re.sub(r'#[^\n]*', '# comentário oculto', intervalo)
        partes.append(html.escape(intervalo))
        partes.append(f'<span title="{token.type}" style="color:{cor_token(token.type)};font-weight:700">'
                      + html.escape(mascarar(token) if ocultar else str(token)) + '</span>')
        cursor = token.end_pos
    final = texto[cursor:]
    if ocultar:
        final = re.sub(r'#[^\n]*', '# comentário oculto', final)
    partes.append(html.escape(final))
    return '<pre class="p1-code">' + ''.join(partes) + '</pre>'


def erro_lexico_html(texto, erro):
    linha = texto.splitlines()[erro.line-1] if texto.splitlines() else ''
    mensagem = f'Erro léxico: linha {erro.line}, coluna {erro.column}, caractere {erro.char!r}.'
    contexto = linha + '\n' + ' ' * (erro.column-1) + '^'
    return '<h4>' + html.escape(mensagem) + '</h4><pre>' + html.escape(contexto) + '</pre><p>' + html.escape(getattr(erro, 'dica', 'Confira o formato do campo.')) + '</p>'


def estatisticas_html(tokens):
    return tabela_html(['Tipo de token', 'Quantidade'], Counter(t.type for t in tokens).most_common())


def resumo_html(tokens, dominio, ocultar=False):
    if dominio == 'a2':
        r = comanda(tokens)
        tabela = tabela_html(['Qtd.', 'Item', 'Unitário', 'Total'],
                            [(q, n, brl(p), brl(q*p)) for q, n, p in r['itens']])
        return tabela + tabela_html(['Campo', 'Resultado'],
                [(chave, brl(r[chave])) for chave in ('subtotal', 'desconto', 'taxa', 'total')]
                + [('Pagamento', r['pagamento']), ('Entrega', r['endereco']),
                   ('Observações', '; '.join(r['observacoes'])), ('Avisos', '; '.join(r['avisos']) or 'Nenhum')])
    if dominio == 'b2':
        inicial, trs = conciliar(tokens)
        saldo, linhas = inicial, []
        for tr in trs:
            saldo += tr['valor']
            chave = (mascarar(tr['chave']) if ocultar else str(tr['chave'])) if tr['chave'] else ''
            linhas.append((tr['data'], tr['hora'], tr['tipo'], chave, tr['descricao'], brl(tr['valor']), brl(saldo)))
        entradas = sum((tr['valor'] for tr in trs if tr['valor'] > 0), Decimal(0))
        saidas = -sum((tr['valor'] for tr in trs if tr['valor'] < 0), Decimal(0))
        return (tabela_html(['Resumo', 'Valor'], [('Saldo inicial', brl(inicial)), ('Entradas', brl(entradas)),
                ('Saídas', brl(saidas)), ('Saldo final', brl(saldo)), ('Transações', len(trs))])
                + tabela_html(['Data', 'Hora', 'Operação', 'Chave', 'Descrição', 'Valor', 'Saldo'], linhas)
                + '<h4>Alertas didáticos</h4><p>' + html.escape(' | '.join(alertas(trs)) or 'Nenhum alerta.') + '</p>')
    if dominio == 'rastreio':
        r = resumo_rastreio(tokens)
        return (tabela_html(['Indicador', 'Resultado'], [('Eventos', len(r['eventos'])), ('Encomendas', r['encomendas']),
                ('Frete por encomenda, última declaração', brl(r['frete_total']))])
                + tabela_html(['Código', 'Status', 'CEP', 'Data/hora', 'Peso (kg)', 'Prazo (dias)'],
                [(e['codigo'], e['status'], e['cep'], e['instante'].strftime('%d/%m/%Y %H:%M'),
                  e.get('peso', ''), e.get('prazo', '')) for e in r['eventos']]))
    chaves = [t for t in tokens if t.type.startswith('CHAVE_')]
    return tabela_html(['Chave reconhecida', 'Valor'], [(t.type, mascarar(t) if ocultar else str(t)) for t in chaves]) if chaves else '<p>Veja a classificação nas abas Colorido e Tokens.</p>'


CSS = '''<style>
.p1-table {border-collapse:collapse; font:14px/1.5 system-ui; width:100%; color:#172033; background:white}
.p1-table th {background:#173047; color:white; text-align:left}
.p1-table td,.p1-table th {padding:7px 11px; border-bottom:1px solid #dce4eb}
.p1-code {white-space:pre-wrap; overflow-wrap:anywhere; padding:18px; background:#f5f8fb; color:#172033; line-height:1.8}
</style>'''


def interface_lexer(titulo, tokenizar, casos, dominio='', laboratorio=False):
    import ipywidgets as widgets
    from IPython.display import display
    seletor = widgets.Dropdown(options=list(casos), description='Casos:', layout=widgets.Layout(width='98%'))
    entrada = widgets.Textarea(value=next(iter(casos.values())), layout=widgets.Layout(width='98%', height='170px'))
    botao = widgets.Button(description='Analisar', button_style='primary', icon='search')
    mascara = widgets.Checkbox(value=dominio in ('b1', 'b2', 'rastreio'), description='Mascarar chaves na saída')
    prioridade = widgets.IntSlider(value=4, min=1, max=5, description='Prior. e-mail:', continuous_update=False)
    status = widgets.HTML()
    paginas = [widgets.HTML() for _ in range(4)]
    abas = widgets.Tab(children=paginas)
    for i, nome in enumerate(['Colorido', 'Tokens', 'Resumo', 'Estatística']):
        abas.set_title(i, nome)

    def atualizar(_=None):
        for pagina in paginas:
            pagina.value = ''
        try:
            tokens = tokenizar(entrada.value, prioridade.value) if laboratorio else tokenizar(entrada.value)
        except UnexpectedCharacters as erro:
            status.value = erro_lexico_html(entrada.value, erro)
            return
        status.value = f'<b>{len(tokens)} tokens reconhecidos.</b> Análise léxica concluída.'
        paginas[0].value = texto_colorido_html(entrada.value, tokens, mascara.value)
        paginas[1].value = tabela_tokens_html(tokens, mascara.value)
        paginas[3].value = estatisticas_html(tokens)
        try:
            paginas[2].value = resumo_html(tokens, dominio, mascara.value)
        except ValueError as erro:
            paginas[2].value = '<b>Erro de estrutura ou de valor no pós-processamento:</b> ' + html.escape(str(erro))

    def escolher(mudanca):
        entrada.value = casos[mudanca['new']]
        atualizar()

    botao.on_click(atualizar)
    seletor.observe(escolher, names='value')
    mascara.observe(atualizar, names='value')
    prioridade.observe(atualizar, names='value')
    elementos = [widgets.HTML(CSS + '<h2>' + html.escape(titulo) + '</h2>'), seletor, entrada,
                 widgets.HBox([botao, mascara])]
    if laboratorio:
        elementos += [prioridade, widgets.HTML('Reservadas: prioridade 3. Compare pix@loja.com.br com prioridade 2 e 4.')]
    elementos += [widgets.HTML('<small>A máscara atua nas chaves reconhecidas da saída. Entrada e contexto de erro continuam editáveis e visíveis.</small>'), status, abas]
    painel = widgets.VBox(elementos)
    atualizar()
    display(painel)
    return dict(painel=painel, entrada=entrada, seletor=seletor, botao=botao, mascara=mascara,
                prioridade=prioridade, status=status, abas=abas, atualizar=atualizar)

# Resolução dos exercícios e experimentos

## 0. Recapitulação

Na linha de montagem de um compilador, o analisador léxico transforma caracteres
em tokens. O parser verifica a organização desses tokens, e a análise semântica
verifica significados e restrições. Só depois vêm etapas como geração e
otimização de código.

Na metáfora do restaurante, o lexer é o funcionário que separa e etiqueta os
ingredientes. Ele reconhece as peças; quem monta o prato é o parser.

- **Token:** categoria, por exemplo `PRECO`.
- **Lexema:** trecho reconhecido, por exemplo `R$ 25,90`.
- **Padrão:** regra que reconhece o trecho, normalmente uma regex.
- **Atributo convertido:** valor útil obtido do lexema, como `Decimal('25.90')`.

Comentários e espaços não são entregues como tokens nas linguagens que usam
`%ignore`, mas suas posições continuam contando para linha e coluna.

## 1. Exercício 1 - Tokenização manual

Entrada com a gramática original A2:

```text
# almoço
pedido 3X "Pastel de Queijo" R$ 8,50
PAGAMENTO Cartão
```

| TOKEN | LEXEMA | LINHA | COLUNA |
|---|---|---:|---:|
| PEDIDO | `pedido` | 2 | 1 |
| QTD | `3X` | 2 | 8 |
| TEXTO | `"Pastel de Queijo"` | 2 | 11 |
| PRECO | `R$ 8,50` | 2 | 30 |
| PAGAMENTO | `PAGAMENTO` | 3 | 1 |
| FORMA_PGTO | `Cartão` | 3 | 11 |

O comentário ocupa a primeira linha, mas não produz token. O modificador `/i`
permite as variações de caixa. As colunas são contadas a partir de 1.

## 2. Exercício 2 - Prioridades no B1 original

| Entrada | Resultado | Justificativa |
|---|---|---|
| `em@banco.com` | `CHAVE_EMAIL` | Prioridade 4 vence a reservada `EM`, de prioridade 3. O `@` permite que `\b` case após `em`, por isso a fronteira sozinha não resolve. |
| `EMPRESA` | Erro em L1, C1, `E` | Depois de `EM` há uma letra, então `em\b` não casa; B1 não tem identificador genérico. |
| `+5511987654321` | `CHAVE_TELEFONE` | Casa com `\+55\d{10,11}`, prioridade 2; os demais padrões não reconhecem esse início completo. |
| `09:45` | `HORA` | Casa com `\d{2}:\d{2}`, prioridade 2. No B1 original não existe token `NUMERO` concorrente. |

## 3. Exercício 3 - Primeiro erro léxico

O erro está na **linha 2, coluna 27, caractere `j`**:

```text
PIX ENVIADO R$ 45,00 PARA joao@@mail.com EM 01/09/2026 10:00
                          ^
```

O e-mail não casa porque o segundo `@` não pertence ao padrão da parte de domínio.
Nenhum outro token do B2 original reconhece o início `j`. O lexer informa a
posição em que não conseguiu começar um token, e não necessariamente a posição
do caractere que um humano identificaria como a causa do erro.

## 4. Exercício 4 - Expressões regulares

```lark
RASTREIO_COD: /[A-Z]{2}\d{9}[A-Z]{2}/
PLACA: /[A-Z]{3}\d[A-Z]\d{2}/
CEP: /\d{5}-\d{3}/
```

- Rastreio: duas letras maiúsculas, nove dígitos, duas letras maiúsculas.
- Placa: três letras, um dígito, uma letra e dois dígitos.
- CEP: cinco dígitos, hífen e três dígitos.

Exemplos válidos: `BR123456789BR`, `ABC1D23` e `01310-100`.
Esses padrões conferem formatos, não a existência dos registros. No teste
isolado usamos `re.fullmatch`, que exige que a entrada inteira corresponda.

## 5. Exercício 5 - Cupom GANHEI100

A existência e a validade do cupom pertencem à **análise semântica**, como regra
de negócio. A sequência `GANHEI100` tem formato de identificador e pode virar
`CODIGO` normalmente. A verificação de existência exige consultar o conjunto de
cupons aceitos. Colocar essa consulta no lexer misturaria reconhecimento de
caracteres com regras que podem mudar sem alterar a linguagem.

Na implementação, `comanda` mantém a tokenização válida, recusa o desconto e
apresenta um aviso. O tokenizador não consulta `CUPONS`.

## 6. Exercícios guiados

### 6.1 A1 - Observações

Acrescentamos o literal `OBS: "OBS"` às definições. A função `compilar` inclui
automaticamente o nome na alternativa `_token`.

```text
PEDIDO 1x "Pizza" R$ 50,00 OBS "sem cebola"
```

Sequência esperada: `PEDIDO QTD ITEM PRECO OBS ITEM`. O A1 continua sensível a
maiúsculas, como o exemplo de partida.

### 6.2 A2 - VR e DESC20

`vr` foi incluído na alternativa de `FORMA_PGTO.3`. `DESC20` foi adicionado à
tabela de cupons com fração `Decimal('0.20')`.

```text
PEDIDO 2x "Pizza" R$ 50,00
TAXA R$ 10,00
CUPOM DESC20
PAGAMENTO vr
```

Subtotal: R$ 100,00. Desconto: R$ 20,00. Taxa: R$ 10,00. **Total: R$ 90,00**.
O desconto incide sobre os itens, antes da taxa. Valores usam `Decimal` e o
desconto é arredondado para centavos, evitando resíduos de ponto flutuante.

### 6.3 B1 - CNPJ

```lark
CHAVE_CNPJ.2: /\d{2}\.\d{3}\.\d{3}\/\d{4}-\d{2}/
```

`12.345.678/0001-90` vira `CHAVE_CNPJ`. O CPF pontuado começa com três dígitos
antes do primeiro ponto, e o CNPJ com dois; os padrões se distinguem pela forma.

### 6.4 B2 - Saque

Adicionamos `SAQUE.3: /saque\b/i` ao lexer e tratamos o saque como débito no
pós-processamento.

```text
SALDO INICIAL R$ 300,00
SAQUE R$ 200,00 EM 05/09/2026 18:00
```

O valor da transação é **-R$ 200,00** e o saldo final é **R$ 100,00**. O resumo
rejeita a estrutura `SAQUE RECEBIDO ...`, mesmo que suas peças sejam tokens válidos.

## 7. Exercícios de nível desafio

### 7.1 A2 - Preço sem R$

```lark
PRECO.2: /(R\$ ?)?\d{1,3}(\.\d{3})*,\d{2}(?![\w,.])/
```

O grupo `(R\$ ?)?` torna o símbolo e seu espaço opcionais. `25,90` e
`R$ 25,90` viram `PRECO`.

**Conflito:** `NUMERO` consegue reconhecer o prefixo `25` de `25,90`. A prioridade
2 de `PRECO` vence a prioridade padrão 0 de `NUMERO`. `QTD.2` mantém prioridade
2; seu padrão exige `x`, enquanto o preço exige vírgula e centavos, portanto
os padrões não reconhecem a mesma sequência completa. Em `25x`, o token é `QTD`.

O lookahead negativo `(?![\w,.])` impede aceitar apenas o início de um preço
com centavos excedentes, como `25,900`. Mantivemos o agrupamento de milhar do
roteiro: use `1.250,00`, e não `1250,00`.

### 7.2 B2 - Telefone sem +55

```lark
CHAVE_TELEFONE.2: /(\+55\d{10,11}|\d{11})(?![\w+])/
```

Aceita `+5511987654321` e `11987654321`. A segunda alternativa tem exatamente
11 dígitos, conforme o exemplo solicitado. Telefone local com 10 dígitos não
foi incluído nesta extensão.

**Nova ambiguidade:** telefone local e CPF sem pontuação podem ter os mesmos
11 dígitos. Não é possível descobrir a intenção somente pela regex. Nesta
linguagem, CPF continua exigindo pontuação e 11 dígitos sem separadores são
classificados como telefone. Assim, `12345678909` também é telefone nesta versão,
enquanto no B2 original produz erro. Isso é uma convenção, não validação cadastral.

Para aceitar os dois formatos sem adivinhação, uma evolução seria exigir um
indicador explícito, como `TIPO CPF` ou `TIPO TELEFONE`, ou fornecer o tipo da
chave em outro campo. Só aumentar prioridades escolheria arbitrariamente um tipo.

### 7.3 B2 - Alerta de chave aleatória

Após a tokenização e conciliação, verificamos conjuntamente:

1. Operação `PIX`.
2. Valor negativo, indicando saída.
3. Chave do tipo `CHAVE_ALEATORIA`.
4. Módulo do valor estritamente maior que `Decimal('500.00')`.

A mensagem é **PIX para chave aleatória acima de R$ 500,00**. Testamos R$ 500,00
(sem alerta), R$ 500,01 (com alerta) e PIX recebido (sem esse alerta). O sinal
serve para revisar o exemplo; não prova irregularidade.

## 8. Experimentos e reflexões do roteiro

### A1: preço sem prioridade

Retirar `.2` de `PRECO` deixa `CODIGO` capturar `R` em `R$ 25,90`; o erro
ocorre no `$`. Restaurar a prioridade faz o preço ser um único token. A célula
de experimento e o teste automático reproduzem as duas situações.

### A1: PEDIDOS e pedido

`PEDIDOS` vira `CODIGO`, porque não é exatamente a reservada literal `PEDIDO`.
`pedido` em minúsculas falha no A1, cujo identificador também exige maiúsculas.
O A2 usa `/i` para as reservadas e aceita identificadores com letras minúsculas.

### A2: por que 8.50 falha no ponto?

Na entrada sem símbolo monetário, `8` pode casar com `NUMERO`. O lexer avança e
encontra `.`, que não inicia nenhum token naquele contexto. Ele não pode prever
que o autor pretendia escrever um preço. A extensão continua exigindo vírgula.

### B1: pix@loja.com.br

Com e-mail na prioridade 4, ele vence `PIX.3`. Ao baixar para 2, `PIX` captura
o início e sobra `@`. O slider da interface reconstrói a gramática para
demonstrar isso. O teste usa a posição do `@` no texto efetivamente analisado.

### B2: resultados do extrato

Saldo inicial R$ 2.500,00; entradas R$ 3.250,00; saídas R$ 1.987,30; saldo final
R$ 3.762,70. São oito transações e seis chaves, abrangendo cinco categorias
léxicas de chave (e-mail, CPF, CNPJ, telefone e aleatória).

O alerta noturno utiliza o intervalo didático de 20h até antes de 6h e valor
estritamente acima de R$ 1.000,00 para PIX de saída. A máscara pode ser ligada
e desligada sem modificar os tokens usados no cálculo.

### Discussão: tokenizador de LLM e lexer de compilador

**Semelhança:** ambos transformam uma sequência de caracteres em unidades
menores que serão consumidas por outra etapa.

**Diferença:** neste compilador, as categorias e padrões são definidos
explicitamente pelo projetista. Em métodos como BPE, o vocabulário é construído
a partir de frequências de sequências em dados. Um token de LLM pode ser só um
fragmento de palavra e não corresponde necessariamente a uma categoria como
`PRECO`. Depois de definido o vocabulário, a tokenização não precisa reaprender
essas unidades a cada texto.

## 9. Entrega final e autoavaliação

O desafio completo está descrito em `linguagem.md`. A autoavaliação individual
fica em `guia_apresentacao.md`, com perguntas para responder antes da entrega.
Ela deve ser marcada pelo estudante após estudar e executar os exemplos.


## Conferência executável da tabela do exercício 1

In [6]:
from IPython.display import display, HTML
entrada_teorica = '# almoço\npedido 3X "Pastel de Queijo" R$ 8,50\nPAGAMENTO Cartão'
display(HTML(CSS + tabela_tokens_html(analisar(entrada_teorica, GRAMATICA_A2_BASE))))


#,TOKEN,LEXEMA,LINHA,COLUNA
1,PEDIDO,pedido,2,1
2,QTD,3X,2,8
3,TEXTO,"""Pastel de Queijo""",2,11
4,PRECO,"R$ 8,50",2,30
5,PAGAMENTO,PAGAMENTO,3,1
6,FORMA_PGTO,Cartão,3,11


## Experimento A1: preço com e sem prioridade

In [7]:
for nome, gramatica in [
    ("Sem prioridade", GRAMATICA_A1_BASE.replace("PRECO.2:", "PRECO:")),
    ("Prioridade corrigida", GRAMATICA_A1_BASE),
]:
    try:
        ts = analisar("R$ 25,90", gramatica)
        print(nome, [(t.type, str(t)) for t in ts])
    except UnexpectedCharacters as e:
        print(f"{nome}: erro na linha {e.line}, coluna {e.column}, caractere {e.char!r}")


Sem prioridade: erro na linha 1, coluna 2, caractere '$'
Prioridade corrigida [('PRECO', 'R$ 25,90')]


## Interface A1 - Comanda com OBS

Escolha um exemplo, edite o texto e clique em **Analisar**.

In [8]:
ui_a1 = interface_lexer('A1 - Comanda com OBS', tokenizar_a1, CASOS_A1, 'a1', laboratorio=False)

## Interface A2 - VR, DESC20 e preço sem R$

Escolha um exemplo, edite o texto e clique em **Analisar**.

In [9]:
ui_a2 = interface_lexer('A2 - VR, DESC20 e preço sem R$', tokenizar_a2, CASOS_A2, 'a2', laboratorio=False)

## Interface B1 - CNPJ e laboratório de prioridade

Escolha um exemplo, edite o texto e clique em **Analisar**.

In [10]:
ui_b1 = interface_lexer('B1 - CNPJ e laboratório de prioridade', tokenizar_b1, CASOS_B1, 'b1', laboratorio=True)

## Interface B2 - Saque, telefone local e alertas

Escolha um exemplo, edite o texto e clique em **Analisar**.

In [11]:
ui_b2 = interface_lexer('B2 - Saque, telefone local e alertas', tokenizar_b2, CASOS_B2, 'b2', laboratorio=False)

# RastreioLang - especificação e diário de ambiguidade

## 1. Cenário

Uma aplicação de logística recebe atualizações de encomendas em texto. O lexer
reconhece comandos, códigos de rastreio, status, CEP, datas, horários, peso,
frete e contatos. O tema corresponde à opção de rastreio de encomendas da P1.

Esta é uma mini-linguagem de estudo; não implementa o protocolo de uma empresa.

## 2. Exemplo completo

```text
# Atualização de encomenda
rastreio AB987654321BR status "em trânsito" cep 20040-020 em 11/09/2026 09:30 origem "São Paulo" destino "Rio de Janeiro" peso 1,250kg frete R$ 25,90 contato em@exemplo.com prazo 3 destinatario 123.456.789-09
```

Cada linha representa um evento. No **pós-processamento**, o cabeçalho obrigatório é:

```text
RASTREIO código STATUS "descrição" CEP número EM data hora
```

Depois do cabeçalho, podem aparecer pares opcionais em qualquer ordem, uma vez
cada: `ORIGEM TEXTO`, `DESTINO TEXTO`, `PESO PESO_VALOR`, `FRETE VALOR`,
`CONTATO EMAIL`, `PRAZO NUMERO`, `DESTINATARIO CPF_VALOR`.
`PRAZO` é uma quantidade inteira de dias e `PESO` está em quilogramas.

## 3. Tabela de tokens

São **22 tipos emitidos**. Comentários e espaços não entram nessa contagem.
As regex abaixo são as usadas em `DEFS_RASTREIO`, incluindo suas restrições
de término. Prioridade omitida no Lark equivale a 0.

| Token | Descrição | Regex Lark | Exemplo | Prioridade |
|---|---|---|---|---:|
| RASTREIO | Início do evento | `/rastreio\b/i` | `rastreio` | 4 |
| STATUS | Rótulo do estado | `/status\b/i` | `STATUS` | 4 |
| CEP | Rótulo do CEP | `/cep\b/i` | `CEP` | 4 |
| EM | Introduz data e hora | `/em\b/i` | `Em` | 4 |
| ORIGEM | Rótulo da origem | `/origem\b/i` | `ORIGEM` | 4 |
| DESTINO | Rótulo do destino | `/destino\b/i` | `DESTINO` | 4 |
| PESO | Rótulo do peso | `/peso\b/i` | `PESO` | 4 |
| FRETE | Rótulo do frete | `/frete\b/i` | `FRETE` | 4 |
| CONTATO | Rótulo do e-mail | `/contato\b/i` | `CONTATO` | 4 |
| PRAZO | Rótulo de dias | `/prazo\b/i` | `PRAZO` | 4 |
| DESTINATARIO | Rótulo do CPF | `/destinatario\b/i` | `DESTINATARIO` | 4 |
| EMAIL | Contato eletrônico | `/[a-z0-9._+-]+@[a-z0-9-]+(\.[a-z0-9-]+)+(?=$\|[\s#])/i` | `em@exemplo.com` | 5 |
| COD_RASTREIO | Código de encomenda | `/[A-Z]{2}\d{9}[A-Z]{2}(?=$\|[\s#])/i` | `BR123456789BR` | 3 |
| CEP_VALOR | CEP pontuado | `/\d{5}-\d{3}(?=$\|[\s#])/` | `01310-100` | 2 |
| DATA | Data no formato dia/mês/ano | `/\d{2}\/\d{2}\/\d{4}(?=$\|[\s#])/` | `10/09/2026` | 2 |
| HORA | Hora e minuto | `/\d{2}:\d{2}(?=$\|[\s#])/` | `08:15` | 2 |
| PESO_VALOR | Peso inteiro ou decimal, com kg | `/\d+(,\d{1,3})?kg(?=$\|[\s#])/i` | `1,250kg` | 2 |
| VALOR | Valor em reais | `/R\$ ?\d{1,3}(\.\d{3})*,\d{2}(?=$\|[\s#])/` | `R$ 25,90` | 2 |
| CPF_VALOR | CPF pontuado | `/\d{3}\.\d{3}\.\d{3}-\d{2}(?=$\|[\s#])/` | `123.456.789-09` | 2 |
| TEXTO | Texto entre aspas | `/"[^"\n]*"/` | `"em trânsito"` | 0 |
| NUMERO | Inteiro não negativo | `/\d+(?=$\|[\s#])/` | `3` | 0 |
| IDENTIFICADOR | Nome genérico | `/[A-Za-z_][A-Za-z0-9_]*(?=$\|[\s#])/` | `LOTE_A` | 0 |

Na tabela Markdown, `\|` escapa a barra vertical para não dividir a célula.
**No arquivo Python e na gramática Lark a alternativa é `|`, sem essa barra
de escape do Markdown.** Para copiar a gramática executável, use `p1.py` ou a
célula do notebook.

### Elementos ignorados

```lark
COMENTARIO: /#[^\n]*/
%ignore COMENTARIO
%ignore /[ \t\r\n]+/
```

Um `#` dentro de um `TEXTO` permanece parte do texto. Fora das aspas, inicia um
comentário até o fim da linha. As posições originais dos tokens são preservadas.

## 4. Como ler os padrões

- `\d`: um dígito; `{9}`: exatamente nove repetições.
- `[A-Z]`: uma letra do intervalo; `/i`: ignora diferença de caixa.
- `\b`: fronteira entre caractere de palavra e não palavra, ou limite da entrada.
  Assim `RASTREIOS` não é dividido na reservada `RASTREIO` e uma letra restante.
- `+`: uma ou mais ocorrências; `*`: zero ou mais; `?`: trecho opcional.
- `\.` e `\$`: ponto e cifrão literais.
- `[^"\n]*`: conteúdo sem aspas e sem quebra de linha.
- `(?=$|[\s#])`: olha adiante, sem consumir, e exige fim da entrada, espaço em
  branco ou começo de comentário. Evita aceitar apenas um prefixo de um campo
  maior malformado, como um CEP seguido de letras.

As 11 reservadas terminam com `\b` e têm `/i`. Datas e CPF conferem o formato;
o lexer não consulta calendário nem calcula dígitos verificadores.

## 5. Diário de ambiguidade

Na implementação, a entrada `CONTATO em@exemplo.com` mostrou um conflito entre
o e-mail e a palavra reservada `EM`. Embora `EM` use `\b`, existe uma fronteira
antes do `@`, então o prefixo ainda pode ser reconhecido como reservada. A
solução foi definir `EMAIL.5` acima das reservadas de prioridade 4. O padrão de
e-mail exige `@` e domínio, portanto `EM` sozinho continua sendo palavra
reservada. Também foi identificado que `BR123456789BR` satisfaz tanto o padrão
específico de rastreio quanto o identificador genérico. `COD_RASTREIO.3` fica
acima de `IDENTIFICADOR`, cuja prioridade é 0. Os testes reduzem essas
prioridades intencionalmente: o primeiro caso produz erro e o segundo passa a
ser identificador, comprovando por que as duas escolhas são necessárias.

### Ordem adotada

```text
5: e-mail
4: palavras reservadas
3: código de rastreio
2: literais estruturados
0: texto, inteiro e identificador
```

O Lark testa primeiro os terminais de maior prioridade. Em empates, considera
o comprimento teórico máximo do padrão, o comprimento da definição e o nome.
Por isso, não tratamos o `basic` como uma busca pelo maior lexema real entre
todos os padrões. Referência: [gramática do Lark](https://lark-parser.readthedocs.io/en/stable/grammar.html).

O identificador genérico serve para demonstrar a diferença entre reconhecimento
léxico e estrutura: `CODIGOERRADO` é uma palavra reconhecível, mas não é um
`COD_RASTREIO`. O resumo recusa seu uso onde espera um código de encomenda.

## 6. Erros e dicas

| Caso | Posição do erro | Explicação e dica |
|---|---|---|
| Status sem fechar aspas | L1 C31, `"` | Fechar o texto do status antes da quebra de linha. |
| CEP `01310.100` | L1 C45, `0` | O CEP exige hífen; nenhum token pode iniciar aquele campo completo. Dica mostra `01310-100`. |
| E-mail `em@@exemplo.com` | L1 C85, `@` | O e-mail completo falha; `EM` é reconhecido antes do `@` remanescente. Dica mostra um e-mail de entrega. |

As posições correspondem às entradas exatas de `CASOS_RASTREIO`. A mensagem
inclui linha, coluna, caractere, contexto com seta e dica. Quando ocorre erro,
as abas anteriores são limpas, para não exibir resultados de outra entrada.

## 7. Casos de teste

- **Válido 1:** evento simples com rastreio, status, CEP, data e hora; 9 tokens.
- **Válido 2:** comentário, reservadas em minúsculas, campos opcionais e o contato
  `em@exemplo.com`; 23 tokens.
- **Válido 3:** três eventos, dois códigos e repetição de frete; 33 tokens,
  duas encomendas e total R$ 35,90.
- **Inválido 1:** aspas abertas no status.
- **Inválido 2:** ponto no lugar do hífen do CEP.
- **Inválido 3:** e-mail com dois sinais `@`.

As entradas completas estão no código, no seletor da interface e no notebook.
Os testes também cobrem as prioridades, as posições, comentários, estrutura
incompleta e campos repetidos.

## 8. Bônus e separação de responsabilidades

`tokenizar_rastreio` somente reconhece tokens. `resumo_rastreio` valida o
cabeçalho e os pares opcionais, converte peso e frete com `Decimal`, prazo com
`int`, data e hora com `datetime`. O resumo conta eventos e encomendas distintas.
Para cada código, vale o último frete informado em ordem cronológica; o mesmo
frete em duas atualizações não é cobrado duas vezes. Em empate de data/hora,
a última declaração na ordem de entrada prevalece.

O mascaramento usa a categoria para esconder parte do e-mail e CPF nas abas
Colorido e Tokens. Os objetos `Token` não são alterados. É uma demonstração de
minimização de exibição, não anonimização de todo o documento: textos livres,
entrada editável e contexto de erro continuam visíveis.


## Desafio: casos de teste e resultados legíveis sem widgets

In [12]:
for nome, texto in CASOS_RASTREIO.items():
    print("\n" + nome + "\n" + texto)
    try:
        ts = tokenizar_rastreio(texto)
        print(f"Resultado: {len(ts)} tokens; {resumo_rastreio(ts)['encomendas']} encomenda(s).")
    except UnexpectedCharacters as e:
        print(f"Erro L{e.line} C{e.column}: {e.char!r}. Dica: {e.dica}")

texto = CASOS_RASTREIO["Válido 2: completo e minúsculas"]
ts = tokenizar_rastreio(texto)
display(HTML(CSS + texto_colorido_html(texto, ts, True)))
display(HTML(tabela_tokens_html(ts, True)))
display(HTML(resumo_html(ts, "rastreio", True)))



Válido 1: entrega
RASTREIO BR123456789BR STATUS "saiu para entrega" CEP 01310-100 EM 10/09/2026 08:15
Resultado: 9 tokens; 1 encomenda(s).

Válido 2: completo e minúsculas
# Encomenda de exemplo
rastreio AB987654321BR status "em trânsito" cep 20040-020 em 11/09/2026 09:30 origem "São Paulo" destino "Rio de Janeiro" peso 1,250kg frete R$ 25,90 contato em@exemplo.com prazo 3 destinatario 123.456.789-09
Resultado: 23 tokens; 1 encomenda(s).

Válido 3: vários eventos
RASTREIO BR123456789BR STATUS "postado" CEP 01310-100 EM 10/09/2026 08:15 FRETE R$ 25,90
RASTREIO BR123456789BR STATUS "entregue" CEP 01310-100 EM 12/09/2026 14:30 FRETE R$ 25,90
RASTREIO CD111222333BR STATUS "postado" CEP 30130-010 EM 12/09/2026 15:00 FRETE R$ 10,00
Resultado: 33 tokens; 2 encomenda(s).

Inválido 1: aspas abertas
RASTREIO BR123456789BR STATUS "saiu para entrega
Erro L1 C31: '"'. Dica: Feche as aspas do nome, status ou descrição antes de mudar de linha.

Inválido 2: CEP com ponto
RASTREIO BR123456789BR STATUS

#,TOKEN,LEXEMA,LINHA,COLUNA
1,RASTREIO,rastreio,2,1
2,COD_RASTREIO,AB987654321BR,2,10
3,STATUS,status,2,24
4,TEXTO,"""em trânsito""",2,31
5,CEP,cep,2,45
6,CEP_VALOR,20040-020,2,49
7,EM,em,2,59
8,DATA,11/09/2026,2,62
9,HORA,09:30,2,73
10,ORIGEM,origem,2,79


## Interface RastreioLang

In [13]:
ui_rastreio = interface_lexer("RastreioLang - Meu Analisador Léxico de Mercado", tokenizar_rastreio, CASOS_RASTREIO, "rastreio")

## Testes automáticos

As verificações abaixo incluem os exercícios teóricos, extensões práticas, desafio e callbacks da interface.

In [14]:
from contextlib import redirect_stdout
from datetime import datetime
from decimal import Decimal
import io
import re
import unittest

from lark import Lark
from lark.exceptions import UnexpectedCharacters
from types import SimpleNamespace
p1 = SimpleNamespace(**globals())


class ExerciciosTeoricos(unittest.TestCase):
    def test_tabela_exercicio_1(self):
        texto = '# almoço\npedido 3X "Pastel de Queijo" R$ 8,50\nPAGAMENTO Cartão'
        ts = p1.analisar(texto, p1.GRAMATICA_A2_BASE)
        self.assertEqual([(t.type, str(t), t.line, t.column) for t in ts], [
            ('PEDIDO', 'pedido', 2, 1), ('QTD', '3X', 2, 8),
            ('TEXTO', '"Pastel de Queijo"', 2, 11), ('PRECO', 'R$ 8,50', 2, 30),
            ('PAGAMENTO', 'PAGAMENTO', 3, 1), ('FORMA_PGTO', 'Cartão', 3, 11)])

    def test_exercicio_2(self):
        for texto, tipo in [('em@banco.com', 'CHAVE_EMAIL'), ('+5511987654321', 'CHAVE_TELEFONE'), ('09:45', 'HORA')]:
            with self.subTest(texto=texto):
                ts = p1.analisar(texto, p1.GRAMATICA_B1_BASE)
                self.assertEqual([(t.type, str(t)) for t in ts], [(tipo, texto)])
        with self.assertRaises(UnexpectedCharacters) as erro:
            p1.analisar('EMPRESA', p1.GRAMATICA_B1_BASE)
        self.assertEqual((erro.exception.line, erro.exception.column, erro.exception.char), (1, 1, 'E'))

    def test_exercicio_3_posicao_real(self):
        texto = 'SALDO INICIAL R$ 300,00\nPIX ENVIADO R$ 45,00 PARA joao@@mail.com EM 01/09/2026 10:00'
        with self.assertRaises(UnexpectedCharacters) as erro:
            p1.analisar(texto, p1.GRAMATICA_B2_BASE)
        self.assertEqual(erro.exception.column, texto.splitlines()[1].index('joao') + 1)
        self.assertEqual((erro.exception.line, erro.exception.column, erro.exception.char), (2, 27, 'j'))

    def test_regex_exercicio_4(self):
        for regex, valido, invalido in [
            (r'[A-Z]{2}\d{9}[A-Z]{2}', 'BR123456789BR', 'BR12345678BR'),
            (r'[A-Z]{3}\d[A-Z]\d{2}', 'ABC1D23', 'ABC1234'),
            (r'\d{5}-\d{3}', '01310-100', '01310100')]:
            with self.subTest(regex=regex):
                self.assertIsNotNone(re.fullmatch(regex, valido))
                self.assertIsNone(re.fullmatch(regex, invalido))

    def test_cupom_inexistente_e_semantica(self):
        r = p1.comanda(p1.tokenizar_a2(p1.CASOS_A2['Cupom inexistente (léxico válido)']))
        self.assertEqual(r['total'], Decimal('32.00'))
        self.assertIn('recusado', r['avisos'][0])


class ExerciciosPraticos(unittest.TestCase):
    def test_a1_obs(self):
        ts = p1.tokenizar_a1(p1.CASOS_A1['Guiado: OBS'])
        self.assertEqual([t.type for t in ts], ['PEDIDO', 'QTD', 'ITEM', 'PRECO', 'OBS', 'ITEM'])

    def test_a1_literal_e_palavra_maior(self):
        self.assertEqual(p1.tokenizar_a1('PEDIDOS')[0].type, 'CODIGO')
        with self.assertRaises(UnexpectedCharacters):
            p1.tokenizar_a1('pedido')

    def test_a1_experimento_preco(self):
        gramatica_ruim = p1.GRAMATICA_A1_BASE.replace('PRECO.2:', 'PRECO:')
        with self.assertRaises(UnexpectedCharacters) as erro:
            p1.analisar('R$ 25,90', gramatica_ruim)
        self.assertEqual((erro.exception.column, erro.exception.char), (2, '$'))
        self.assertEqual(p1.tokenizar_a1('R$ 25,90')[0].type, 'PRECO')

    def test_vr_e_desc20_com_calculo(self):
        ts = p1.tokenizar_a2(p1.CASOS_A2['Guiado: VR e DESC20'])
        self.assertEqual(ts[-1].type, 'FORMA_PGTO')
        r = p1.comanda(ts)
        self.assertEqual((r['subtotal'], r['desconto'], r['total']), (Decimal(100), Decimal(20), Decimal(90)))

    def test_preco_sem_simbolo_contra_numero_e_quantidade(self):
        ts = p1.tokenizar_a2('25,90 25 25x R$ 1.250,00 1.250,00')
        self.assertEqual([t.type for t in ts], ['PRECO', 'NUMERO', 'QTD', 'PRECO', 'PRECO'])
        with self.assertRaises(UnexpectedCharacters):
            p1.tokenizar_a2('25,900')

    def test_b1_cnpj_e_cpf(self):
        self.assertEqual(p1.tokenizar_b1('12.345.678/0001-90')[0].type, 'CHAVE_CNPJ')
        self.assertEqual(p1.tokenizar_b1('123.456.789-09')[0].type, 'CHAVE_CPF')

    def test_b1_prioridade_email(self):
        self.assertEqual(p1.tokenizar_b1('pix@loja.com.br', 4)[0].type, 'CHAVE_EMAIL')
        with self.assertRaises(UnexpectedCharacters) as erro:
            p1.tokenizar_b1('pix@loja.com.br', 2)
        self.assertEqual((erro.exception.column, erro.exception.char), (4, '@'))

    def test_saque_sempre_saida(self):
        inicial, trs = p1.conciliar(p1.tokenizar_b2(p1.CASOS_B2['Guiado: SAQUE']))
        self.assertEqual(trs[0]['valor'], Decimal('-200.00'))
        self.assertEqual(inicial+trs[0]['valor'], Decimal('100.00'))
        with self.assertRaises(ValueError):
            p1.conciliar(p1.tokenizar_b2('SAQUE RECEBIDO R$ 200,00 EM 05/09/2026 18:00'))

    def test_telefone_local_e_versoes(self):
        self.assertEqual(p1.tokenizar_b2('11987654321')[0].type, 'CHAVE_TELEFONE')
        for gramatica in (p1.GRAMATICA_B1_BASE, p1.GRAMATICA_B2_BASE):
            with self.assertRaises(UnexpectedCharacters):
                p1.analisar('11987654321', gramatica)
        self.assertEqual(p1.tokenizar_b2('12345678909')[0].type, 'CHAVE_TELEFONE')
        for invalido in ('119876543210', '11987654321abc'):
            with self.assertRaises(UnexpectedCharacters):
                p1.tokenizar_b2(invalido)

    def test_alerta_aleatoria_limiar_e_direcao(self):
        for valor, direcao, esperado in [('500,00', 'ENVIADO', False), ('500,01', 'ENVIADO', True), ('900,00', 'RECEBIDO', False)]:
            with self.subTest(valor=valor, direcao=direcao):
                prep = 'PARA' if direcao == 'ENVIADO' else 'DE'
                texto = f'PIX {direcao} R$ {valor} {prep} {p1.UUID_EXEMPLO} EM 10/09/2026 14:32'
                _, trs = p1.conciliar(p1.tokenizar_b2(texto))
                self.assertEqual(bool(p1.alertas(trs)), esperado)

    def test_alerta_noturno_fronteiras(self):
        for hora, esperado in [('05:59', True), ('06:00', False), ('19:59', False), ('20:00', True)]:
            texto = f'PIX ENVIADO R$ 1.000,01 PARA a@example.com EM 10/09/2026 {hora}'
            _, trs = p1.conciliar(p1.tokenizar_b2(texto))
            self.assertEqual(bool(p1.alertas(trs)), esperado)

    def test_comanda_e_extrato_do_roteiro(self):
        r = p1.comanda(p1.tokenizar_a2(p1.CASOS_A2['Pedido completo']))
        self.assertEqual(r['total'], Decimal('87.46'))
        r = p1.comanda(p1.tokenizar_a2(p1.CASOS_A2['Frete grátis']))
        self.assertEqual((r['total'], r['taxa']), (Decimal('1150.00'), Decimal(0)))
        inicial, trs = p1.conciliar(p1.tokenizar_b2(p1.EXTRATO_EXEMPLO))
        self.assertEqual(sum((t['valor'] for t in trs if t['valor'] > 0), Decimal(0)), Decimal('3250.00'))
        self.assertEqual(-sum((t['valor'] for t in trs if t['valor'] < 0), Decimal(0)), Decimal('1987.30'))
        self.assertEqual(inicial+sum(t['valor'] for t in trs), Decimal('3762.70'))

    def test_data_inexistente_apenas_no_pos_processamento(self):
        ts = p1.tokenizar_b2('SAQUE R$ 20,00 EM 31/02/2026 18:00')
        self.assertIn('DATA', [t.type for t in ts])
        with self.assertRaises(ValueError):
            p1.conciliar(ts)


class DesafioRastreio(unittest.TestCase):
    def test_quantidade_tokens_e_reservadas(self):
        lexer = p1.criar_lexer(p1.GRAMATICA_RASTREIO)
        terminais = [t for t in lexer.terminals if t.name not in lexer.ignore_tokens]
        self.assertEqual(len(terminais), 22)
        reservadas = [t for t in terminais if t.priority == 4]
        self.assertEqual(len(reservadas), 11)
        for terminal in reservadas:
            self.assertIn('i', terminal.pattern.flags)
            self.assertIn(r'\b', terminal.pattern.value)

    def test_tres_validos(self):
        for nome, texto in p1.CASOS_RASTREIO.items():
            if nome.startswith('Válido'):
                with self.subTest(nome=nome):
                    ts = p1.tokenizar_rastreio(texto)
                    self.assertGreater(len(ts), 0)
                    self.assertGreater(p1.resumo_rastreio(ts)['encomendas'], 0)

    def test_invalidos_posicoes_e_dicas(self):
        esperados = [(1, 31, '"'), (1, 45, '0'), (1, 85, '@')]
        invalidos = [v for k, v in p1.CASOS_RASTREIO.items() if k.startswith('Inválido')]
        for texto, esperado in zip(invalidos, esperados):
            with self.subTest(texto=texto):
                with self.assertRaises(UnexpectedCharacters) as ctx:
                    p1.tokenizar_rastreio(texto)
                e = ctx.exception
                self.assertEqual((e.line, e.column, e.char), esperado)
                self.assertGreater(len(e.dica), 30)

    def test_conflitos_documentados(self):
        self.assertEqual(p1.tokenizar_rastreio('em@exemplo.com')[0].type, 'EMAIL')
        self.assertEqual(p1.tokenizar_rastreio('BR123456789BR')[0].type, 'COD_RASTREIO')
        ruim = p1.GRAMATICA_RASTREIO.replace('EMAIL.5:', 'EMAIL.3:')
        with self.assertRaises(UnexpectedCharacters):
            p1.analisar('em@exemplo.com', ruim)
        ruim = p1.GRAMATICA_RASTREIO.replace('COD_RASTREIO.3:', 'COD_RASTREIO:')
        self.assertEqual(p1.analisar('BR123456789BR', ruim)[0].type, 'IDENTIFICADOR')

    def test_fronteira_de_palavra(self):
        ts = p1.tokenizar_rastreio('RASTREIOS Em empresa 3 3kg')
        self.assertEqual([t.type for t in ts], ['IDENTIFICADOR', 'EM', 'IDENTIFICADOR', 'NUMERO', 'PESO_VALOR'])

    def test_comentarios_e_posicoes(self):
        texto = '# início\n\trastreio BR123456789BR # fim\nSTATUS "# dentro do texto"'
        ts = p1.tokenizar_rastreio(texto)
        self.assertEqual((ts[0].line, ts[0].column), (2, 2))
        self.assertEqual(str(ts[-1]), '"# dentro do texto"')
        for t in ts:
            self.assertEqual(texto[t.start_pos:t.end_pos], str(t))

    def test_conversoes_e_frete_sem_duplicar(self):
        r = p1.resumo_rastreio(p1.tokenizar_rastreio(p1.CASOS_RASTREIO['Válido 2: completo e minúsculas']))
        e = r['eventos'][0]
        self.assertEqual(e['peso'], Decimal('1.250'))
        self.assertEqual(e['prazo'], 3)
        self.assertIsInstance(e['instante'], datetime)
        r = p1.resumo_rastreio(p1.tokenizar_rastreio(p1.CASOS_RASTREIO['Válido 3: vários eventos']))
        self.assertEqual((r['encomendas'], len(r['eventos']), r['frete_total']), (2, 3, Decimal('35.90')))

    def test_lexico_nao_garante_estrutura(self):
        for texto in ('STATUS RASTREIO', p1.CASOS_RASTREIO['Válido 1: entrega'].replace('BR123456789BR', 'CODIGOERRADO')):
            ts = p1.tokenizar_rastreio(texto)
            with self.assertRaises(ValueError):
                p1.resumo_rastreio(ts)

    def test_rejeita_campos_duplicados(self):
        texto = p1.CASOS_RASTREIO['Válido 1: entrega'] + ' FRETE R$ 1,00 FRETE R$ 2,00'
        with self.assertRaises(ValueError):
            p1.resumo_rastreio(p1.tokenizar_rastreio(texto))


class Apresentacao(unittest.TestCase):
    def test_escape_html(self):
        texto = '"<script>alert(1)</script>"'
        ts = p1.tokenizar_rastreio(texto)
        for saida in (p1.texto_colorido_html(texto, ts), p1.tabela_tokens_html(ts)):
            self.assertNotIn('<script>', saida)
            self.assertIn('&lt;script&gt;', saida)

    def test_mascara_preserva_tokens(self):
        texto = 'CONTATO em@exemplo.com DESTINATARIO 123.456.789-09 # anotação'
        ts = p1.tokenizar_rastreio(texto)
        antes = [(str(t), t.start_pos, t.end_pos) for t in ts]
        for saida in (p1.texto_colorido_html(texto, ts, True), p1.tabela_tokens_html(ts, True)):
            self.assertNotIn('em@exemplo.com', saida)
            self.assertNotIn('123.456.789-09', saida)
        self.assertEqual(antes, [(str(t), t.start_pos, t.end_pos) for t in ts])

    def test_interfaces_callbacks_e_limpeza(self):
        for dominio, funcao, casos in [('a1', p1.tokenizar_a1, p1.CASOS_A1), ('a2', p1.tokenizar_a2, p1.CASOS_A2),
                                       ('b1', p1.tokenizar_b1, p1.CASOS_B1), ('b2', p1.tokenizar_b2, p1.CASOS_B2),
                                       ('rastreio', p1.tokenizar_rastreio, p1.CASOS_RASTREIO)]:
            with self.subTest(dominio=dominio), redirect_stdout(io.StringIO()):
                ui = p1.interface_lexer(dominio, funcao, casos, dominio, dominio == 'b1')
                for nome in casos:
                    ui['seletor'].value = nome
                    ui['botao'].click()
                    self.assertTrue(ui['status'].value)
                    if nome.startswith('Inválido'):
                        self.assertIn('Erro léxico', ui['status'].value)
                        self.assertTrue(all(not pagina.value for pagina in ui['abas'].children))
                    else:
                        self.assertIn('tokens reconhecidos', ui['status'].value)
                ui['mascara'].value = not ui['mascara'].value
                if dominio == 'b1':
                    ui['seletor'].value = 'Conflito pix@'
                    ui['prioridade'].value = 2
                    self.assertIn('Erro léxico', ui['status'].value)
                    ui['prioridade'].value = 4
                    self.assertIn('tokens reconhecidos', ui['status'].value)
                ui['painel'].close()



suite = unittest.TestSuite()
for classe in (ExerciciosTeoricos, ExerciciosPraticos, DesafioRastreio, Apresentacao):
    suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(classe))
resultado = unittest.TextTestRunner(verbosity=2).run(suite)
assert resultado.wasSuccessful(), "Há falhas a corrigir antes da entrega."


test_cupom_inexistente_e_semantica (__main__.ExerciciosTeoricos.test_cupom_inexistente_e_semantica) ... 

ok


test_exercicio_2 (__main__.ExerciciosTeoricos.test_exercicio_2) ... 

ok


test_exercicio_3_posicao_real (__main__.ExerciciosTeoricos.test_exercicio_3_posicao_real) ... 

...

test_interfaces_callbacks_e_limpeza (__main__.Apresentacao.test_interfaces_callbacks_e_limpeza) ... 

ok


test_mascara_preserva_tokens (__main__.Apresentacao.test_mascara_preserva_tokens) ... 

ok


----------------------------------------------------------------------
Ran 30 tests in 0.203s

OK


# Guia de apresentação e estudo

## Demonstração sugerida (5 a 7 minutos)

1. **Objetivo (30 s).** “O projeto transforma o texto de pedidos, transações e
   rastreios em tokens. Cada token tem categoria, lexema, linha e coluna.”
2. **RastreioLang (1 min).** Abra o primeiro caso válido. Mostre `RASTREIO`,
   `COD_RASTREIO`, `TEXTO`, `CEP_VALOR`, `DATA` e `HORA` nas duas primeiras abas.
3. **Prioridade (1 min).** No B1, escolha o caso `pix@`, reduza a prioridade
   para 2 e mostre o erro no `@`. Volte para 4. Explique a relação com `EM`
   e `EMAIL` do desafio.
4. **Erros (1 min).** No desafio, selecione aspas abertas e CEP com ponto.
   Mostre linha, coluna, seta e dica específica.
5. **Extensões (1 min).** Mostre VR/DESC20, saque e alerta para chave aleatória.
6. **Bônus (1 min).** Abra o caso com três eventos, confira dois códigos e
   R$ 35,90 de frete. Ligue/desligue a máscara no caso completo.
7. **Testes (30 s).** Execute a célula de testes e explique um caso de fronteira:
   R$ 500,00 não dispara o alerta; R$ 500,01 dispara.

## Perguntas que você precisa saber responder

**Por que `start: _token*`?**  
Para aceitar qualquer quantidade de tokens enquanto estudamos só o lexer.
Essa regra não exige a estrutura de um comando completo.

**Qual é a diferença entre token e lexema?**  
`CEP_VALOR` é o tipo; `01310-100` é o texto concreto reconhecido.

**O que significa `/i`?**  
Aceitar letras maiúsculas e minúsculas no padrão.

**Por que usar `\b`?**  
Para não reconhecer uma palavra reservada apenas como começo de uma palavra
maior. `EM` é reservada; `EMPRESA` não é.

**Por que `\b` não basta para um e-mail?**  
Porque `@` não é caractere de palavra. Existe uma fronteira após `em` em
`em@exemplo.com`, então precisamos dar prioridade maior ao e-mail completo.

**O que `.5` significa?**  
Prioridade léxica 5. Um padrão que casa nessa prioridade é testado antes de
outro com prioridade 4. Não indica quantidade de caracteres ou repetições.

**Como você mostra linha e coluna?**  
Os tokens do Lark já trazem esses atributos. A exceção `UnexpectedCharacters`
também os fornece quando a leitura falha.

**Como o texto é colorido sem perder espaços?**  
`start_pos` e `end_pos` delimitam cada lexema no texto original. A interface
preserva os intervalos entre tokens e colore somente o trecho reconhecido.

**O que os widgets fazem?**  
`Textarea` recebe a entrada; `Button.on_click` chama a análise;
`Dropdown.observe` reage à escolha de exemplo. As abas usam widgets `HTML`
atualizados pelo callback. `Output`, apresentado no roteiro, é outra forma de
exibir saídas de `display`, mas não é necessário para estas abas de HTML.

**Uma data reconhecida necessariamente existe?**  
Não. A regex verifica o formato. A conversão com `datetime` rejeita valores
como 31 de fevereiro na etapa posterior.

**Como distinguir telefone local e CPF sem pontuação?**  
Somente 11 dígitos não bastam. Nossa convenção exige pontuação no CPF e trata
11 dígitos puros como telefone. Um campo explícito de tipo resolveria a intenção.

**Por que não usar `float` para dinheiro?**  
`Decimal` permite representar os valores decimais dos exemplos sem resíduos
binários. A regra de arredondamento do desconto é definida explicitamente.

## Autoavaliação individual

Marque depois de executar e explicar cada item com suas palavras:

- [ ] Diferencio token, lexema e padrão.
- [ ] Escrevo um token com literal e outro com regex.
- [ ] Explico `%ignore` e a preservação de linha e coluna.
- [ ] Demonstro e resolvo um conflito de prioridade.
- [ ] Explico `/i` e `\b`, inclusive a limitação com `@`.
- [ ] Identifico e trato `UnexpectedCharacters`.
- [ ] Explico `line`, `column`, `start_pos` e `end_pos`.
- [ ] Explico os componentes da interface e seus callbacks.
- [ ] Distingo erro léxico, erro de estrutura e regra de negócio.
- [ ] Consigo ler cada regex da tabela do desafio em voz alta.
- [ ] Executei o notebook do início ao fim no meu ambiente.



## Referências e entrega

- Material da disciplina: *Aula CP 05 (Prática 1) - Construindo um Analisador Léxico com Lark*, Prof. Hercules Ramos, FMU, 2026. As gramáticas A e B foram adaptadas desse roteiro.
- [Documentação do Lark](https://lark-parser.readthedocs.io/en/stable/grammar.html).
- [Documentação do ipywidgets](https://ipywidgets.readthedocs.io/en/stable/).

Publique o projeto com README em um repositório público no GitHub e envie o
link na atividade até **25/09/2026 às 23h59**. Confira os nomes e a composição
do grupo. A publicação não efetua o envio no ambiente da faculdade.
